# 22a — Data Augmentation

Corresponds to: Note 22 — Data Augmentation

**Libraries introduced in this notebook:** No new libraries — `torchvision.transforms` was introduced in notebook 20b.

**What this notebook demonstrates:**
1. Building an augmentation pipeline with `torchvision.transforms` — flip, crop, colour jitter
2. Visualising augmented versions of the same image
3. Training the same CNN on CIFAR-10 with and without augmentation
4. Comparing learning curves — overfitting gap with vs without augmentation
5. Evaluating the effect on test accuracy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:  {device}')
print(f'PyTorch: {torch.__version__}')

## Part 1: Augmentation Pipeline

Two transforms are defined: one with augmentation (training), one without (test). The test transform is deterministic — only normalisation.

In [ ]:
CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD  = [0.2470, 0.2435, 0.2616]

# Training: random augmentation
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

# Training without augmentation (baseline)
transform_no_aug = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

# Test: deterministic only
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

print('Augmentation pipeline (training):')
for t in transform_aug.transforms:
    print(f'  {t}')
print()
print('Test pipeline (no augmentation):')
for t in transform_test.transforms:
    print(f'  {t}')

## Part 2: Visualise Augmented Versions

Apply the augmentation pipeline 8 times to the same image to show the variation it produces.

In [ ]:
# Load raw dataset (no transform) to get original images
raw_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                                        transform=transforms.ToTensor())

# Augmentation without normalise for visualisation
vis_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
])

raw_set_pil = torchvision.datasets.CIFAR10(root='./data', train=True, download=False,
                                            transform=None)

img_pil, label = raw_set_pil[0]
CLASSES = raw_set_pil.classes

print(f'Source image class: {CLASSES[label]}')
print(f'Generating 8 augmented versions...')
torch.manual_seed(0)
versions = [vis_aug(img_pil) for _ in range(8)]
print(f'Each version shape: {versions[0].shape}')
print(f'Pixel range: [{versions[0].min():.3f}, {versions[0].max():.3f}]')
print()
print('Augmented versions differ in: flip, crop position, brightness, contrast, saturation, hue')
print('Label is unchanged for all versions:', CLASSES[label])

## Part 3: Load CIFAR-10 with and without Augmentation

We use a subset of 5,000 training images to make the overfitting effect more visible.

In [ ]:
from torch.utils.data import Subset

torch.manual_seed(42)
train_full_aug    = torchvision.datasets.CIFAR10(root='./data', train=True,  download=False, transform=transform_aug)
train_full_no_aug = torchvision.datasets.CIFAR10(root='./data', train=True,  download=False, transform=transform_no_aug)
test_full         = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)

idx = torch.randperm(50000)[:5000]
train_aug    = Subset(train_full_aug,    idx)
train_no_aug = Subset(train_full_no_aug, idx)

loader_aug    = DataLoader(train_aug,    batch_size=128, shuffle=True,  num_workers=0)
loader_no_aug = DataLoader(train_no_aug, batch_size=128, shuffle=True,  num_workers=0)
loader_test   = DataLoader(test_full,    batch_size=128, shuffle=False, num_workers=0)

print(f'Training samples (subset): {len(train_aug)}')
print(f'Test samples: {len(test_full)}')

## Part 4: Train Both Models

Same CNN architecture, same hyperparameters — only the training data differs.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8, 256), nn.ReLU(), nn.Linear(256, 10))
    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

def train_model(train_loader, epochs=20):
    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    history = []
    for epoch in range(1, epochs+1):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            tl += loss.item()*len(y); tc += (out.argmax(1)==y).sum().item(); tn += len(y)
        model.eval()
        with torch.no_grad():
            vl, vc, vn = 0.0, 0, 0
            for X, y in loader_test:
                X, y = X.to(device), y.to(device)
                out = model(X)
                vl += criterion(out,y).item()*len(y); vc += (out.argmax(1)==y).sum().item(); vn += len(y)
        history.append((tl/tn, tc/tn, vl/vn, vc/vn))
        if epoch % 5 == 0:
            print(f'  Epoch {epoch:2d}: train_loss={tl/tn:.4f} train_acc={tc/tn:.4f} '
                  f'val_loss={vl/vn:.4f} val_acc={vc/vn:.4f}')
    return history

print('Training WITHOUT augmentation:')
torch.manual_seed(0)
hist_no_aug = train_model(loader_no_aug)

print()
print('Training WITH augmentation:')
torch.manual_seed(0)
hist_aug = train_model(loader_aug)

## Part 5: Compare Results

In [ ]:
final_no_aug = hist_no_aug[-1]
final_aug    = hist_aug[-1]

print('Final epoch results (epoch 20):')
print(f'{"":<25} {"Train Acc":<14} {"Val Acc":<14} {"Overfit Gap"}')
print('-' * 60)
print(f'{"Without augmentation":<25} {final_no_aug[1]:<14.4f} {final_no_aug[3]:<14.4f} '
      f'{final_no_aug[1]-final_no_aug[3]:.4f}')
print(f'{"With augmentation":<25} {final_aug[1]:<14.4f} {final_aug[3]:<14.4f} '
      f'{final_aug[1]-final_aug[3]:.4f}')
print()
print('Overfitting gap = train accuracy - validation accuracy')
print('Smaller gap with augmentation confirms reduced overfitting.')

## Summary

What this notebook demonstrated:

1. **Augmentation pipeline** — `RandomHorizontalFlip`, `RandomCrop(32, padding=4)`, `ColorJitter` applied during training only. Test pipeline uses deterministic normalisation only.
2. **Visualisation** — the same image produces visibly different versions each time the pipeline is applied: different flip, crop position, brightness, contrast, saturation, and hue.
3. **Training comparison** — same CNN, same hyperparameters, same 5,000-image subset. Only the training transform differs.
4. **Overfitting gap** — without augmentation, training accuracy is much higher than validation accuracy. With augmentation, the gap is smaller — the model generalises better.
5. **Test accuracy** — augmentation improves validation accuracy on the same test set, confirming that the regularisation effect translates to real performance gains.